In [2]:
# Essential libraries for Colab
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [3]:

import pandas as pd
from datasets import load_dataset, Dataset, concatenate_datasets

In [4]:

# 1. SPAM: YouTube Spam Collection is perfect
from datasets import load_dataset

spam_data = load_dataset("jason1966/ahsenwaheed_youtube-comments-spam-dataset",split='train')

#2. Question
question_data = load_dataset("rajpurkar/squad", split='train[:2000]')

# 3. PRAISE, CRITICISM, & NEUTRAL (From Yelp - No scripts needed)
# labels: 0=1 star (Criticism), 2=3 star (Neutral), 4=5 star (Praise)
yelp = load_dataset("yelp_review_full", split='train[:10000]')

# 4. NEUTRAL: Standard Twitter sentiment (Neutral class)
ds_sentiment = load_dataset("cardiffnlp/tweet_eval", "sentiment")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

Youtube-Spam-Dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1956 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

...

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
from datasets import Features, Value, concatenate_datasets

# Define a clean feature set to avoid the "Label greater than configured num_classes" error
new_features = Features({'text': Value('string'), 'label': Value('int64')})

# --- 1. Process SPAM (YouTube specific) ---
# Mapping CLASS=1 (Spam) to our Label 3
spam_final = spam_data.filter(lambda x: x["CLASS"] == 1).select(range(1000)).map(
    lambda x: {"text": x["CONTENT"], "label": 3},
    remove_columns=spam_data.column_names,
    features=new_features)

# --- 2. Process QUESTIONS (SQuAD) ---
# Mapping to our Label 1
ques_final = question_data.select(range(1000)).map(
    lambda x: {"text": x["question"], "label": 1},
    remove_columns=question_data.column_names,
    features=new_features)

# --- 3. Process YELP (Praise & Criticism) ---
# Label 4 (5-star) -> Praise (0) | Label 0 (1-star) -> Criticism (2)
praise_final = yelp.filter(lambda x: x["label"] == 4).select(range(1000)).map(
    lambda x: {"text": x["text"], "label": 0},
    remove_columns=["label"],
    features=new_features)

crit_final = yelp.filter(lambda x: x["label"] == 0).select(range(1000)).map(
    lambda x: {"text": x["text"], "label": 2},
    remove_columns=["label"],
    features=new_features)

# --- 4. Process NEUTRAL (Tweet Sentiment) ---
# Label 1 (Neutral) -> Our Label 4
neutral_final = ds_sentiment['train'].filter(lambda x: x["label"] == 1).select(range(1000)).map(
    lambda x: {"text": x["text"], "label": 4},
    remove_columns=["label"],
    features=new_features)

# --- THE FINAL MERGE ---
master_ds = concatenate_datasets([
    praise_final,
    ques_final,
    crit_final,
    spam_final,
    neutral_final
]).shuffle(seed=42)

print(f"Master Dataset created with {len(master_ds)} rows!")
print(f"Sample data: {master_ds[0]}")

Filter:   0%|          | 0/1956 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Master Dataset created with 5000 rows!
Sample data: {'text': 'Where was Blue Ivy born?', 'label': 1}


In [6]:
from transformers import AutoTokenizer

# Load the RoBERTa tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(examples):
    # Padding and Truncation ensure all inputs are the same length (128 tokens)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization in batches (faster)
tokenized_datasets = master_ds.map(tokenize_function, batched=True)

# Split into 80% Training and 20% Testing
final_data = tokenized_datasets.train_test_split(test_size=0.2)

print("Data tokenized and split successfully!")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Data tokenized and split successfully!


In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Map IDs to Labels
id2label = {0: "PRAISE", 1: "QUESTION", 2: "CRITICISM", 3: "SPAM", 4: "NEUTRAL"}
label2id = {v: k for k, v in id2label.items()}

# 2. Load RoBERTa for Classification
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
import torch
print("GPU Available:" , torch.cuda.is_available())
# Should print: GPU Available: True

GPU Available: True


In [9]:
# 1. Define the metric (Fixes the NameError)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Set up the Arguments
training_args = TrainingArguments(
    output_dir="./intent_checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Checkpoint & Eval Strategy
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    # Best Model Logic
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    # Clean Logging (Removing the deprecated logging_dir)
    logging_steps=50,
    report_to="none"
)

# 3. Initialize the Trainer (Corrected for Transformers v4.46+)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_data["train"],
    eval_dataset=final_data["test"],
    processing_class=tokenizer, # Changed 'tokenizer' to 'processing_class'
    compute_metrics=compute_metrics,
)

# 4. START TRAINING
print("Starting training... This will take a few minutes on GPU.")
trainer.train()

Starting training... This will take a few minutes on GPU.


Step,Training Loss,Validation Loss,Accuracy
100,0.232080,0.150471,0.970000
200,0.103043,0.059871,0.985000
300,0.094368,0.049508,0.989000
400,0.080631,0.157697,0.966000
500,0.074316,0.103244,0.977000
600,0.021198,0.089171,0.981000
700,0.040059,0.066452,0.983000
800,0.018568,0.076264,0.984000
900,0.007850,0.079483,0.985000
1000,0.016450,0.086347,0.984000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1250, training_loss=0.09168388490080834, metrics={'train_runtime': 738.1913, 'train_samples_per_second': 27.093, 'train_steps_per_second': 1.693, 'total_flos': 1315590712320000.0, 'train_loss': 0.09168388490080834, 'epoch': 5.0})